# 02 Exploratory Data Analysis

This notebook performs exploratory data analysis (EDA) for `data/CardiacPatientData.csv` and saves report-ready figures to `reports/figures/`.

The analysis includes cohort summaries, outcome balance, outcome-stratified comparisons, feature distributions, risk-category outcome rates, and numeric-predictor correlations. The raw CSV is read but not modified.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300

# Resolve paths so the notebook runs from either the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "CardiacPatientData.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME_COL = "Outcome"
ID_COL = "ID"

# Feature groupings used throughout the EDA.
VITAL_FEATURES = ["SBP", "DBP", "HR", "RR", "BT", "SpO2"]
LAB_FEATURES = ["Na", "K", "Cl", "Urea", "Ceratinine"]  # Preserve raw column spelling.
SCORE_FEATURES = ["GCS", "TriageScore"]
DEMOGRAPHIC_FEATURES = ["Age"]
BINARY_FEATURES = ["Smoke", "Alcoholic", "FHCD", "Gender"]
NUMERIC_PREDICTORS = DEMOGRAPHIC_FEATURES + VITAL_FEATURES + LAB_FEATURES + SCORE_FEATURES
IMPORTANT_NUMERIC_FEATURES = ["Age", "SBP", "DBP", "HR", "RR", "SpO2", "GCS", "Na", "K", "Urea", "Ceratinine", "TriageScore"]


def save_current_figure(filename: str) -> Path:
    # Save the active matplotlib figure to reports/figures/ with tight layout.
    output_path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches="tight")
    print(f"Saved figure: {output_path.relative_to(PROJECT_ROOT)}")
    return output_path

print(f"Data path: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Figures directory: {FIGURES_DIR.relative_to(PROJECT_ROOT)}")


## Load data

Read the source CSV into memory and inspect the first rows before producing summary tables or figures.


In [ ]:
df = pd.read_csv(DATA_PATH)

# Keep category columns easy to read in tables and plots while preserving the raw values.
df["OutcomeLabel"] = df[OUTCOME_COL].map({0: "Outcome 0", 1: "Outcome 1"}).fillna(df[OUTCOME_COL].astype(str))
df["SmokeLabel"] = df["Smoke"].map({0: "No", 1: "Yes"}).fillna("Missing")
df["AlcoholicLabel"] = df["Alcoholic"].map({0: "No", 1: "Yes"}).fillna("Missing")
df["FHCDLabel"] = df["FHCD"].map({0: "No", 1: "Yes"}).fillna("Missing")
df["GenderLabel"] = df["Gender"].map({0: "Gender 0", 1: "Gender 1"}).fillna("Missing")

display(df.head())
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")


The dataset contains one row per recorded observation and includes demographics, vital signs, laboratory values, clinical scores, history indicators, and the binary `Outcome` label. The `ID` column may represent repeated observations for the same patient or encounter, so it is summarized separately from clinical predictors.


## 1. Overall cohort summary

Summarize cohort size, unique IDs, missingness, numeric distributions, and binary/categorical distributions.


In [ ]:
cohort_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns_in_raw_csv",
            "unique_ids",
            "duplicated_full_rows",
            "rows_with_any_missing_value",
        ],
        "value": [
            len(df),
            len(pd.read_csv(DATA_PATH, nrows=0).columns),
            df[ID_COL].nunique(dropna=True),
            df.drop(columns=[c for c in df.columns if c.endswith("Label")], errors="ignore").duplicated().sum(),
            df.drop(columns=[c for c in df.columns if c.endswith("Label")], errors="ignore").isna().any(axis=1).sum(),
        ],
    }
)
display(cohort_summary)

numeric_summary = df[NUMERIC_PREDICTORS].describe().T
numeric_summary["missing"] = df[NUMERIC_PREDICTORS].isna().sum()
numeric_summary["missing_pct"] = df[NUMERIC_PREDICTORS].isna().mean() * 100
display(numeric_summary.round(2))

categorical_summary = []
for col in BINARY_FEATURES + ["Outcome"]:
    counts = df[col].value_counts(dropna=False).sort_index()
    rates = df[col].value_counts(dropna=False, normalize=True).sort_index() * 100
    for value in counts.index:
        categorical_summary.append({"feature": col, "value": value, "count": counts.loc[value], "percent": rates.loc[value]})

display(pd.DataFrame(categorical_summary).round({"percent": 2}))


The overall summary provides a baseline for the analytic cohort and highlights variables with missing values before stratified comparisons are made. Numeric predictors span demographics, vitals, labs, GCS, and triage score; the raw `Ceratinine` spelling is retained to preserve lineage to the source file.


## 2. Outcome class balance

Estimate the size and proportion of each outcome class and save a bar chart for the report.


In [ ]:
outcome_balance = (
    df[OUTCOME_COL]
    .value_counts(dropna=False)
    .rename_axis("Outcome")
    .reset_index(name="count")
    .sort_values("Outcome")
)
outcome_balance["percent"] = outcome_balance["count"] / outcome_balance["count"].sum() * 100
display(outcome_balance.round({"percent": 2}))

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=outcome_balance, x="Outcome", y="count", hue="Outcome", palette="Set2", legend=False, ax=ax)
ax.set_title("Outcome Class Balance")
ax.set_xlabel("Outcome class")
ax.set_ylabel("Number of observations")
for patch, pct in zip(ax.patches, outcome_balance["percent"]):
    ax.annotate(f"{pct:.1f}%", (patch.get_x() + patch.get_width() / 2, patch.get_height()), ha="center", va="bottom")
save_current_figure("eda_02_outcome_class_balance.png")
plt.show()


The classes are imbalanced, with the positive outcome class representing most observations. Model development should therefore report metrics that are robust to imbalance, such as sensitivity, specificity, precision-recall metrics, and calibrated probabilities, rather than relying only on accuracy.


## 3. Cohort summary stratified by Outcome

Compare central tendency and spread for numeric predictors across outcome classes.


In [ ]:
stratified_summary = df.groupby(OUTCOME_COL)[NUMERIC_PREDICTORS].agg(["count", "mean", "std", "median", "min", "max"])
display(stratified_summary.round(2))

binary_by_outcome = []
for col in BINARY_FEATURES:
    table = pd.crosstab(df[col], df[OUTCOME_COL], margins=False, dropna=False)
    rate_table = pd.crosstab(df[col], df[OUTCOME_COL], normalize="columns", dropna=False) * 100
    for value in table.index:
        row = {"feature": col, "value": value}
        for outcome in table.columns:
            row[f"Outcome {outcome} count"] = table.loc[value, outcome]
            row[f"Outcome {outcome} column_pct"] = rate_table.loc[value, outcome]
        binary_by_outcome.append(row)

display(pd.DataFrame(binary_by_outcome).round(2))


Outcome-stratified summaries show which features differ most between classes before formal modeling. In this cohort, several vitals and oxygenation-related measures differ by outcome, while `Gender` appears similar across outcome groups; these are descriptive comparisons and do not establish independent predictors.


## 4. Histograms / KDE plots for age, vitals, labs, GCS, and triage score

Plot univariate distributions for the main numeric predictors. These figures help identify skew, multimodality, outliers, and coding artifacts.


In [ ]:
distribution_groups = {
    "age": DEMOGRAPHIC_FEATURES,
    "vitals": VITAL_FEATURES,
    "labs": LAB_FEATURES,
    "scores": SCORE_FEATURES,
}

for group_name, features in distribution_groups.items():
    n_cols = 3
    n_rows = int(np.ceil(len(features) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 3.8 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, feature in zip(axes, features):
        sns.histplot(data=df, x=feature, kde=True, bins=30, color="#4C78A8", ax=ax)
        ax.set_title(f"Distribution of {feature}")
        ax.set_xlabel(feature)
        ax.set_ylabel("Number of observations")
    for ax in axes[len(features):]:
        ax.set_visible(False)
    save_current_figure(f"eda_04_distribution_{group_name}.png")
    plt.show()


The distribution plots should be reviewed for clinically implausible values and heavy skew before modeling. Laboratory variables and some vitals may have long tails, while score variables such as GCS and triage score are discrete and may be concentrated at a small number of values.


## 5. Boxplots comparing important numeric features by Outcome

Use boxplots to compare important numeric features by outcome class. These plots emphasize medians, interquartile ranges, and outliers.


In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(IMPORTANT_NUMERIC_FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 4.0 * n_rows))
axes = axes.ravel()
for ax, feature in zip(axes, IMPORTANT_NUMERIC_FEATURES):
    sns.boxplot(data=df, x=OUTCOME_COL, y=feature, hue=OUTCOME_COL, palette="Set2", legend=False, showfliers=False, ax=ax)
    ax.set_title(f"{feature} by Outcome")
    ax.set_xlabel("Outcome class")
    ax.set_ylabel(feature)
for ax in axes[len(IMPORTANT_NUMERIC_FEATURES):]:
    ax.set_visible(False)
save_current_figure("eda_05_numeric_boxplots_by_outcome.png")
plt.show()


The boxplots suggest that age, hemodynamics, respiratory rate, oxygen saturation, and selected lab markers may differ across outcomes. Outliers are hidden in this report-facing view to make median and IQR differences easier to compare; outlier-specific checks should be handled in the data quality workflow.


## 6. Outcome rates by age bands

Group age into clinically interpretable bands and calculate the outcome rate in each band.


In [ ]:
age_bins = [0, 39, 49, 59, 69, 79, np.inf]
age_labels = ["<40", "40-49", "50-59", "60-69", "70-79", "80+"]
df["AgeBand"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels, include_lowest=True, right=True)

age_band_rates = (
    df.groupby("AgeBand", observed=False)[OUTCOME_COL]
    .agg(count="count", outcome_rate="mean")
    .reset_index()
)
age_band_rates["outcome_rate_pct"] = age_band_rates["outcome_rate"] * 100
display(age_band_rates.round({"outcome_rate": 3, "outcome_rate_pct": 1}))

fig, ax = plt.subplots(figsize=(8, 4.8))
sns.barplot(data=age_band_rates, x="AgeBand", y="outcome_rate_pct", color="#59A14F", ax=ax)
ax.set_title("Outcome Rate by Age Band")
ax.set_xlabel("Age band (years)")
ax.set_ylabel("Outcome rate (%)")
ax.set_ylim(0, 100)
for patch, count in zip(ax.patches, age_band_rates["count"]):
    ax.annotate(f"n={count:,}", (patch.get_x() + patch.get_width() / 2, patch.get_height()), ha="center", va="bottom", fontsize=9)
save_current_figure("eda_06_outcome_rate_by_age_band.png")
plt.show()


Outcome rates vary by age band, which suggests that age may carry clinically useful risk information. Because repeated IDs may exist, these rates describe observations rather than necessarily unique patients unless the source confirms one row per patient.


## 7. Outcome rates by SpO2 category

Categorize oxygen saturation into clinically interpretable bands and compare outcome rates.


In [ ]:
spo2_bins = [-np.inf, 89, 94, 100, np.inf]
spo2_labels = ["<90%", "90-94%", "95-100%", ">100% / check coding"]
df["SpO2Category"] = pd.cut(df["SpO2"], bins=spo2_bins, labels=spo2_labels, right=True)

spo2_rates = (
    df.groupby("SpO2Category", observed=False)[OUTCOME_COL]
    .agg(count="count", outcome_rate="mean")
    .reset_index()
)
spo2_rates["outcome_rate_pct"] = spo2_rates["outcome_rate"] * 100
display(spo2_rates.round({"outcome_rate": 3, "outcome_rate_pct": 1}))

fig, ax = plt.subplots(figsize=(8, 4.8))
sns.barplot(data=spo2_rates, x="SpO2Category", y="outcome_rate_pct", color="#E15759", ax=ax)
ax.set_title("Outcome Rate by SpO2 Category")
ax.set_xlabel("SpO2 category")
ax.set_ylabel("Outcome rate (%)")
ax.set_ylim(0, 100)
for patch, count in zip(ax.patches, spo2_rates["count"]):
    ax.annotate(f"n={count:,}", (patch.get_x() + patch.get_width() / 2, patch.get_height()), ha="center", va="bottom", fontsize=9)
save_current_figure("eda_07_outcome_rate_by_spo2_category.png")
plt.show()


Lower oxygen saturation categories show visibly different outcome rates than normal-range saturation. Any values above 100%, if present, should be treated as potential coding or device-entry issues rather than clinically valid saturation percentages.


## 8. Outcome rates by GCS severity category

Convert GCS into common severity categories and compare outcome rates by neurologic status.


In [ ]:
gcs_bins = [-np.inf, 8, 12, 15, np.inf]
gcs_labels = ["Severe (3-8)", "Moderate (9-12)", "Mild/Normal (13-15)", ">15 / check coding"]
df["GCSSeverity"] = pd.cut(df["GCS"], bins=gcs_bins, labels=gcs_labels, right=True)

gcs_rates = (
    df.groupby("GCSSeverity", observed=False)[OUTCOME_COL]
    .agg(count="count", outcome_rate="mean")
    .reset_index()
)
gcs_rates["outcome_rate_pct"] = gcs_rates["outcome_rate"] * 100
display(gcs_rates.round({"outcome_rate": 3, "outcome_rate_pct": 1}))

fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(data=gcs_rates, x="GCSSeverity", y="outcome_rate_pct", color="#B07AA1", ax=ax)
ax.set_title("Outcome Rate by GCS Severity Category")
ax.set_xlabel("GCS severity category")
ax.set_ylabel("Outcome rate (%)")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=20)
for patch, count in zip(ax.patches, gcs_rates["count"]):
    ax.annotate(f"n={count:,}", (patch.get_x() + patch.get_width() / 2, patch.get_height()), ha="center", va="bottom", fontsize=9)
save_current_figure("eda_08_outcome_rate_by_gcs_severity.png")
plt.show()


Most observations cluster in the mild/normal GCS range, so sparse categories should be interpreted cautiously. Values above 15 are not valid on the standard GCS scale and should be reviewed as possible coding artifacts before clinical conclusions are drawn.


## 9. Outcome rates by smoking, alcohol, family history, and gender

Compare outcome rates across binary history indicators and encoded gender groups.


In [ ]:
category_specs = {
    "Smoke": "SmokeLabel",
    "Alcoholic": "AlcoholicLabel",
    "FHCD": "FHCDLabel",
    "Gender": "GenderLabel",
}

rate_tables = []
for raw_col, label_col in category_specs.items():
    rates = (
        df.groupby(label_col, dropna=False)[OUTCOME_COL]
        .agg(count="count", outcome_rate="mean")
        .reset_index()
        .rename(columns={label_col: "category"})
    )
    rates["feature"] = raw_col
    rates["outcome_rate_pct"] = rates["outcome_rate"] * 100
    rate_tables.append(rates[["feature", "category", "count", "outcome_rate", "outcome_rate_pct"]])

history_gender_rates = pd.concat(rate_tables, ignore_index=True)
display(history_gender_rates.round({"outcome_rate": 3, "outcome_rate_pct": 1}))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
for ax, (raw_col, label_col) in zip(axes, category_specs.items()):
    plot_df = history_gender_rates[history_gender_rates["feature"] == raw_col]
    sns.barplot(data=plot_df, x="category", y="outcome_rate_pct", color="#F28E2B", ax=ax)
    ax.set_title(f"Outcome Rate by {raw_col}")
    ax.set_xlabel(raw_col)
    ax.set_ylabel("Outcome rate (%)")
    ax.set_ylim(0, 100)
    for patch, count in zip(ax.patches, plot_df["count"]):
        ax.annotate(f"n={count:,}", (patch.get_x() + patch.get_width() / 2, patch.get_height()), ha="center", va="bottom", fontsize=9)
save_current_figure("eda_09_outcome_rates_history_gender.png")
plt.show()


Smoking, alcohol status, and family history show descriptive differences in outcome rates, while the encoded gender groups appear similar. The meaning and direction of the `Gender` codes are not documented here, so subgroup interpretation should remain code-based until source metadata confirms the mapping.


## 10. Correlation heatmap for numeric predictors

Calculate Pearson correlations among numeric predictors, excluding `ID` and `Outcome` to avoid treating identifiers or labels as predictors.


In [ ]:
corr = df[NUMERIC_PREDICTORS].corr(method="pearson")

display(corr.round(2))

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)
ax.set_title("Correlation Heatmap for Numeric Predictors")
save_current_figure("eda_10_numeric_predictor_correlation_heatmap.png")
plt.show()


The heatmap identifies correlated predictors that may affect model stability and feature interpretation. Strong clinical pairings, such as systolic and diastolic blood pressure or related electrolyte measures, should be considered when selecting model families and interpreting feature importance.


## Figure inventory

List the plots saved by this EDA notebook for use in reports and downstream review.


In [ ]:
saved_eda_figures = sorted(FIGURES_DIR.glob("eda_*.png"))
figure_inventory = pd.DataFrame(
    {"figure": [path.name for path in saved_eda_figures], "relative_path": [str(path.relative_to(PROJECT_ROOT)) for path in saved_eda_figures]}
)
display(figure_inventory)


The saved figure inventory provides a compact checklist of report-ready EDA outputs. If this notebook is re-run after data updates, the files in `reports/figures/` with the `eda_` prefix will be refreshed with the latest plots.


## Conclusion

This EDA shows that the cardiac arrest dataset contains useful clinical signal for later modeling. Outcome groups differ across several important variables, including age, vital signs, GCS, triage score, oxygen saturation, and selected lab values. The class imbalance also shows that future models should not be judged by accuracy alone, but with metrics like recall, F1 score, ROC-AUC, PR-AUC, and calibration.

Before modeling, the dataset should be cleaned for implausible values, repeated IDs should be reviewed, and categorical or risk-group features should be encoded properly. Overall, this notebook supports moving into feature engineering and baseline predictive modeling while keeping the analysis clinically interpretable.